# Internal Bar Strength and Exponential Moving Averages

This daily long/cash backtest on AAPL enters when IBS is at most 0.25 and the 8-day EMA is below the 24-day EMA. It exits when IBS is at least 0.75 and the 8-day EMA is above the 24-day EMA.

The example downloads daily prices from Yahoo Finance through yfinance. Signals and simulated fills use the same closing price. The backtest uses fractional shares, liquidates any open position at the end of the sample, and omits commissions and slippage. Amounts are in US dollars.


In [ ]:
from _market_data import download_ohlcv


def calculate_ibs(data):
    price_range = data["High"] - data["Low"]
    return (data["Close"] - data["Low"]).div(price_range.where(price_range > 0))


def calculate_ema(data, window):
    return data["Close"].ewm(span=window, adjust=False).mean()


def backtest_strategy(data, initial_capital=2000):
    capital = initial_capital
    shares = 0.0
    trade_cost = 0.0
    trades = []

    for row in data.itertuples():
        if row.Buy_Signal and shares == 0:
            trade_cost = capital
            shares = capital / row.Close
            capital = 0.0
            trades.append({"Type": "Buy", "Date": row.Index, "Price": row.Close})
        elif row.Sell_Signal and shares > 0:
            capital = shares * row.Close
            shares = 0.0
            trades.append({
                "Type": "Sell",
                "Date": row.Index,
                "Price": row.Close,
                "Gain/Loss": capital - trade_cost,
            })

    if shares > 0:
        close = data["Close"].iloc[-1]
        capital = shares * close
        trades.append({
            "Type": "Sell",
            "Date": data.index[-1],
            "Price": close,
            "Gain/Loss": capital - trade_cost,
        })

    return capital - initial_capital, trades

In [ ]:
start_date = "2011-01-01"
end_date = "2023-08-06"
initial_capital = 2000

data = download_ohlcv("AAPL", start_date, end_date, end_inclusive=False)
data["IBS"] = calculate_ibs(data)
data["EMA_8"] = calculate_ema(data, window=8)
data["EMA_24"] = calculate_ema(data, window=24)
data["Buy_Signal"] = (data["IBS"] <= 0.25) & (data["EMA_8"] < data["EMA_24"])
data["Sell_Signal"] = (data["IBS"] >= 0.75) & (data["EMA_8"] > data["EMA_24"])

total_gain_loss, trades = backtest_strategy(data, initial_capital)
print(f"Total profit or loss: {total_gain_loss:,.2f} USD")
print("\nTrades:")
for trade in trades:
    details = f"{trade['Type']} | {trade['Date']:%Y-%m-%d} | {trade['Price']:.2f} USD"
    if "Gain/Loss" in trade:
        details += f" | Profit or loss: {trade['Gain/Loss']:,.2f} USD"
    print(details)